# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Review the available record sets, their `@id`s, and the fields they contain. This helps in understanding the structure of the dataset and selecting data for further analysis.

In [ ]:
# List all available record sets and their fields by @id
record_set_ids = []
print("Available record sets and their fields (referenced by @id):\n")
for record_set in dataset.record_sets:
    print(f"Record set: {record_set['@id']} (name: {record_set.get('name', '')})")
    record_set_ids.append(record_set['@id'])
    if 'field' in record_set:
        fields = record_set['field']
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            print(f"  - Field: {field['@id']} (name: {field.get('name', '')}, dataType: {field.get('dataType', '')})")
    print("")
print("All record set @id's:")
print(record_set_ids)

## 3. Data Extraction

Load data from one or more record sets into Pandas DataFrames for further processing. Use the record set and field `@id`s from the overview above. Each entity is referenced strictly by its `@id` for clarity and reproducibility.

In [ ]:
# Extract data for each record set by @id

# (You may change which record sets to include depending on the output above.)
record_sets_to_load = record_set_ids  # Load all by default
dataframes = {}

for record_set_id in record_sets_to_load:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for {record_set_id} with {len(df)} rows and columns:")
            print(df.columns.tolist())
        else:
            print(f"Record set {record_set_id} returned no records.")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

print("\nPreview of the first DataFrame loaded:")
# Show a preview of the first loaded dataframe (if any)
if dataframes:
    first_id = next(iter(dataframes))
    display(dataframes[first_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)

Let's perform some basic data processing, such as filtering records on a numeric field, normalizing, or grouping data. We operate with field `@id`s for full traceability.

In [ ]:
# Identify a suitable DataFrame, numeric field, and group field for EDA.
# You may need to adapt the chosen fields based on the previous overview.

# Select the first non-empty DataFrame
if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id}")

    # Try to find a likely numeric field (@id ending in or containing 'age', 'interval', 'count', 'duration', etc.)
    candidate_numeric_fields = [col for col in df.columns if any(
        word in col.lower() for word in ['age', 'interval', 'count', 'duration', 'years', 'length', 'number', 'metastasis']
    )]
    if candidate_numeric_fields:
        numeric_field_id = candidate_numeric_fields[0]
        print(f"Selected numeric field: {numeric_field_id}")
        # Convert to numeric if needed
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        # Example filter: keep values greater than the median
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with '{numeric_field_id}' > {threshold}")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to find a candidate categorical field ending in 'sex', 'gender', 'site', or 'msi'
        candidate_group_fields = [col for col in df.columns if any(word in col.lower() for word in ['sex', 'gender', 'site', 'msi', 'location', 'type'])]
        if candidate_group_fields:
            group_field_id = candidate_group_fields[0]
            print(f"Grouping records by '{group_field_id}': mean {numeric_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No obvious numeric field found for EDA in this record set.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization

Visualization helps reveal insights in the data more intuitively. We will plot distributions or relationships between fields as available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic matplotlib/seaborn setup
if dataframes and 'filtered_df' in locals():
    # Numeric field distribution
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} in filtered records")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

    # If group field exists, show boxplot
    if 'group_field_id' in locals():
        plt.figure(figsize=(8,5))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- This notebook demonstrated how to load, overview, and explore a FAIR-compliant Croissant dataset with `mlcroissant`.
- All dataset entities (record sets, fields, and groupings) were referenced by their Croissant `@id` for reproducibility.
- We loaded tabular data on clinicopathological cancer survivor characteristics, performed basic filtering, normalization, grouping, and exploratory visualization.

You can further extend this notebook to perform advanced statistical analysis, modeling, or other domain-specific tasks. For more information, see the [`mlcroissant` documentation](https://mlcroissant.readthedocs.io/).